## Setup the data

In [2]:
import os
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import time
import numpy as np
from mobile_sam import sam_model_registry, SamPredictor

/opt/conda/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/opt/conda/lib/python3.12/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/opt/conda/lib/python3.12/site-packages/mobile_sam/modeling/tiny_vit_sam.py:656: UserWarning: Overwriting tiny_vit_5m_224 in registry with mobile_sam.modeling.tiny_vit_sam.tiny_vit_5m_224. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  return register_model(fn_wrapper)
/opt/conda/lib/python3.12/site-packages/mobile_sam/modeling/tiny_vit_sam.py:656: UserWarning: Overwriting tiny_vit_11m

## Instance the model (not compiled)

In [8]:
device = torch.device("cpu")

base_ckpt = "immich-sticker-gen/serving/models/mobile_sam.pt"
model_path = "immich-sticker-gen/serving/models/distilled_mobile_sam_full.pth"

model = sam_model_registry["vit_t"](checkpoint=base_ckpt)
state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

predictor = SamPredictor(model)

/tmp/ipykernel_248/3667006114.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path, map_location=device)


## Load test data from SA-1B

In [9]:
SHARD_URL = "https://scontent.xx.fbcdn.net/m1/v/t6/An_YmP5OIPXun-vu3hkckAZZ2s4lPYoVkiyvCcWiVY21mu1Ng5_1HeCa2CWiSTsskj8HQ8bN013HxNpYDdSC_7jWQq_svcg.tar?_nc_gid&ccb=10-5&oh=00_Af08ZycNdYdNMlMCW5sVXhQ0W2iYlA4GO1vtsjm6IY-yYw&oe=69F57028&_nc_sid=0fdd51"

import os
import json
import requests
import tarfile
from pathlib import Path
from PIL import Image

def prepare_sa1b_subset(
    shard_url,
    out_dir="benchmark_sa1b",
    max_images=50,
    tar_path="sa1b_shard.tar",
):
    out_dir = Path(out_dir)
    images_dir = out_dir / "images"
    anns_dir = out_dir / "annotations"
    images_dir.mkdir(parents=True, exist_ok=True)
    anns_dir.mkdir(parents=True, exist_ok=True)

    # download once
    if not os.path.exists(tar_path):
        print("Downloading shard...")
        with requests.get(shard_url, stream=True) as r:
            r.raise_for_status()
            with open(tar_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
        print("Download complete.")
    else:
        print("Shard already exists, skipping download.")

    manifest_path = out_dir / "manifest.json"
    if manifest_path.exists():
        with open(manifest_path, "r") as f:
            manifest = json.load(f)
        if len(manifest) >= max_images:
            print(f"Subset already exists ({len(manifest)} pairs), skipping extraction.")
            return manifest[:max_images]

    print("Extracting image/json pairs...")
    saved = []
    pending_json = {}

    with tarfile.open(tar_path) as tar:
        members = tar.getmembers()

        # first pass: read jsons into memory by stem
        for member in members:
            if member.isfile() and member.name.endswith(".json"):
                stem = Path(member.name).stem
                f = tar.extractfile(member)
                if f is not None:
                    pending_json[stem] = f.read()

        count = 0

        # second pass: extract matching jpg + json
        for member in members:
            if not (member.isfile() and member.name.endswith(".jpg")):
                continue

            stem = Path(member.name).stem
            if stem not in pending_json:
                continue

            f = tar.extractfile(member)
            if f is None:
                continue

            img = Image.open(f).convert("RGB")

            img_path = images_dir / f"{stem}.jpg"
            ann_path = anns_dir / f"{stem}.json"

            img.save(img_path, quality=95)

            with open(ann_path, "wb") as jf:
                jf.write(pending_json[stem])

            saved.append({
                "id": stem,
                "image_path": str(img_path),
                "annotation_path": str(ann_path),
            })

            count += 1
            if count >= max_images:
                break

    with open(manifest_path, "w") as f:
        json.dump(saved, f, indent=2)

    print(f"Saved {len(saved)} image/json pairs to {out_dir}")
    return saved

pairs = prepare_sa1b_subset(
    shard_url=SHARD_URL,
    out_dir="benchmark_sa1b",
    max_images=50,
    tar_path="sa1b_shard.tar",
)

Download complete.
Extracting image/json pairs...
Saved 50 image/json pairs to benchmark_sa1b


## Calculate metrics

In [10]:
import os
import json
import time
import numpy as np
import torch
import cv2
from PIL import Image
from pycocotools import mask as mask_utils

# assumes these already exist:
# model
# predictor
# model_path
# pairs = prepare_sa1b_subset(...)

# ----------------------------
# helpers
# ----------------------------
def load_pair(pair):
    image = np.array(Image.open(pair["image_path"]).convert("RGB"))
    with open(pair["annotation_path"], "r") as f:
        ann_data = json.load(f)
    return image, ann_data

def ann_to_mask(ann):
    seg = ann["segmentation"]
    if isinstance(seg["counts"], list):
        rle = mask_utils.frPyObjects(seg, seg["size"][0], seg["size"][1])
    else:
        rle = seg
    mask = mask_utils.decode(rle)
    if mask.ndim == 3:
        mask = mask[..., 0]
    return mask.astype(bool)

def mask_to_box(mask):
    ys, xs = np.where(mask)
    return np.array([xs.min(), ys.min(), xs.max(), ys.max()], dtype=np.float32)

def compute_iou(pred_mask, gt_mask):
    inter = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()
    return inter / union if union > 0 else 0.0

PIXEL_MEAN = torch.tensor([123.675, 116.28, 103.53]).view(3, 1, 1)
PIXEL_STD  = torch.tensor([58.395, 57.12, 57.375]).view(3, 1, 1)

def preprocess_sam(image_rgb, image_size=1024):
    h, w = image_rgb.shape[:2]
    scale = image_size / max(h, w)
    new_h, new_w = int(h * scale + 0.5), int(w * scale + 0.5)

    resized = cv2.resize(image_rgb, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    pad_h = image_size - new_h
    pad_w = image_size - new_w
    padded = np.pad(resized, ((0, pad_h), (0, pad_w), (0, 0)), mode="constant")

    x = torch.from_numpy(padded).permute(2, 0, 1).float()
    x = (x - PIXEL_MEAN) / PIXEL_STD
    return x

# ----------------------------
# config
# ----------------------------
device = next(model.parameters()).device
num_eval_samples = min(50, len(pairs))
num_trials = 100
num_batches = 50
batch_size = min(8, len(pairs))

eval_pairs = pairs[:num_eval_samples]

model_size = os.path.getsize(model_path)
print(f"Model Size on Disk: {model_size / 1e6:.2f} MB")

# ----------------------------
# accuracy = mean IoU
# ----------------------------
ious = []

with torch.no_grad():
    for pair in eval_pairs:
        image, ann_data = load_pair(pair)

        anns = ann_data.get("annotations", [])
        if len(anns) == 0:
            continue

        # use first GT mask
        gt_mask = ann_to_mask(anns[0])
        box = mask_to_box(gt_mask)

        predictor.set_image(image)
        masks, scores, logits = predictor.predict(
            box=box[None, :],
            multimask_output=False
        )

        pred_mask = masks[0].astype(bool)
        iou = compute_iou(pred_mask, gt_mask)
        ious.append(iou)

mean_iou = 100.0 * np.mean(ious) if ious else 0.0
print(f"Mean IoU: {mean_iou:.2f}% ({len(ious)} samples)")

# ----------------------------
# single-sample latency
# ----------------------------
single_image, single_ann_data = load_pair(eval_pairs[0])
single_gt_mask = ann_to_mask(single_ann_data["annotations"][0])
single_box = mask_to_box(single_gt_mask)

with torch.no_grad():
    predictor.set_image(single_image)
    predictor.predict(
        box=single_box[None, :],
        multimask_output=False
    )

latencies = []
with torch.no_grad():
    for _ in range(num_trials):
        start_time = time.time()
        predictor.set_image(single_image)
        _ = predictor.predict(
            box=single_box[None, :],
            multimask_output=False
        )
        latencies.append(time.time() - start_time)

print(f"Inference Latency (single sample, median): {np.percentile(latencies, 50) * 1000:.2f} ms")
print(f"Inference Latency (single sample, 95th percentile): {np.percentile(latencies, 95) * 1000:.2f} ms")
print(f"Inference Latency (single sample, 99th percentile): {np.percentile(latencies, 99) * 1000:.2f} ms")
print(f"Inference Throughput (single sample): {num_trials / np.sum(latencies):.2f} FPS")

# ----------------------------
# batch throughput (encoder only)
# ----------------------------
batch_tensors = []
for pair in eval_pairs[:batch_size]:
    image, _ = load_pair(pair)
    batch_tensors.append(preprocess_sam(image))

batch_input = torch.stack(batch_tensors).to(device)

with torch.no_grad():
    model.image_encoder(batch_input)

batch_times = []
with torch.no_grad():
    for _ in range(num_batches):
        start_time = time.time()
        _ = model.image_encoder(batch_input)
        batch_times.append(time.time() - start_time)

batch_fps = (batch_input.shape[0] * num_batches) / np.sum(batch_times)
print(f"Batch Throughput (encoder): {batch_fps:.2f} FPS")
print(f"Batch Latency (encoder, median): {np.percentile(batch_times, 50) * 1000:.2f} ms")

Model Size on Disk: 40.73 MB
Mean IoU: 83.13% (50 samples)
Inference Latency (single sample, median): 248.86 ms
Inference Latency (single sample, 95th percentile): 277.97 ms
Inference Latency (single sample, 99th percentile): 285.99 ms
Inference Throughput (single sample): 3.95 FPS
Batch Throughput (encoder): 2.16 FPS
Batch Latency (encoder, median): 3691.26 ms
